# 11 — How much can the data tell us? Sharpe inference

Notebooks 08–10 found no method that beats EW, no CAPM α that survives the
search, and no overlay that changes a Sharpe ratio detectably. This notebook
asks whether 17 years of daily data could have found such differences at all.
It is descriptive: no hypotheses, no verdicts.

It applies López de Prado, Lipton & Zoonekynd (2026), "How to Use the Sharpe
Ratio", implemented in `maplab.sharpe`. The paper treats an estimated Sharpe
ratio as a noisy statistic and gives the tools to judge it: a variance for the
estimate that allows for skewed, fat-tailed and autocorrelated returns; the
probabilistic Sharpe ratio (PSR), the confidence that the true Sharpe ratio
exceeds a threshold; the minimum track record length (MinTRL), the history
needed for that confidence; the power of the test; and the deflated Sharpe
ratio (DSR), which discounts the best result of a search for the number of
trials behind it. Equation numbers below refer to that paper.

## Setup

Run labels as in notebook 08.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import maplab as ml
import _nb11_helpers as h
from maplab import apply_style

apply_style()

## §1 What we ask

Fixed in `registrations/nb11.toml` after an exploratory run of the same data and
formulas, so it records the design rather than pre-registering it. Data: daily
net returns in excess of BIL for the eight core runs and 60/40, and for the 23
runs of notebooks 01–07, in notebook 08's full, train and test windows (D1 uses
all three, D2 full and test, D3 and D4 the full window).

| | Question | Measure |
|---|---|---|
| D1 | How sure are we that each run beats cash, and how many years would its Sharpe ratio need to show it? | PSR against 0 and MinTRL, α = 0.05 |
| D2 | How often would a true annual Sharpe ratio of 0.25, 0.5 or 0.75 be detected against cash? | Power, and years to 80% power |
| D3 | How often could notebook 08's H1 tests detect a true annual gap of 0.1, 0.2 or 0.3 against EW? | Power with notebook 08's standard errors, unadjusted and at α/7; years to 80% power |
| D4 | What is the best of the 23 runs worth once the search is accounted for? | DSR for K = 23 and for an effective number of independent trials (clustering seed 20261002), with a sharp-null variance as a sensitivity |

D1, D2 and D4 follow the paper; D3 is our own normal-approximation calculation,
not the paper's. Limits: PSR and DSR test against cash, not against EW; the
variance formula assumes first-order autocorrelation; the effective K is an
estimate.

In [ ]:
reg = ml.inference.load_registration("nb11")
assert [x["id"] for x in reg["hypothesis"]] == ["D1", "D2", "D3", "D4"]
assert all(x["kind"] == "descriptive" for x in reg["hypothesis"])

## §2 The formulas

Per observation, with $\gamma_3$ the skewness, $\gamma_4$ the Pearson kurtosis (3 for normal returns) and $\rho$ the lag-1 autocorrelation of the returns:

- Variance of the estimated Sharpe ratio (eq. 2–3 and 5): $\sigma^2_{SR} = \dfrac{1}{T}\left[\dfrac{1+\rho}{1-\rho} - \dfrac{1+\rho+\rho^2}{1-\rho^2}\,\gamma_3\,SR + \dfrac{1+\rho^2}{1-\rho^2}\,\dfrac{\gamma_4-1}{4}\,SR^2\right]$, which is $(1 + SR^2/2)/T$ for iid Normal returns.
- Probabilistic Sharpe ratio (eq. 9): $PSR = \Phi\!\left(\dfrac{\widehat{SR}-SR_0}{\sigma_0}\right)$, with $\sigma_0^2$ the variance above evaluated at $SR_0$.
- Minimum track record length (eq. 11): $MinTRL = \sigma_0^2\big|_{T=1}\left(\dfrac{\Phi^{-1}(1-\alpha)}{\widehat{SR}-SR_0}\right)^2$ observations.
- Power (eq. 15–16): $SR_c = SR_0 + \Phi^{-1}(1-\alpha)\,\sigma_0$ and $1-\beta = \Phi\!\left(\dfrac{SR_1 - SR_c}{\sigma_1}\right)$, with $\sigma_1^2$ evaluated at $SR_1$.
- Expected maximum and deflated Sharpe ratio (eq. 28–30): $SR_{0,K} = SR_0 + \sqrt{V}\left[(1-\gamma)\,\Phi^{-1}\!\left(1-\tfrac1K\right) + \gamma\,\Phi^{-1}\!\left(1-\tfrac{1}{Ke}\right)\right]$ ($\gamma$ Euler–Mascheroni, $V$ the variance of the $K$ trials' Sharpe ratios), $s_{0,K} = \sqrt{V}\,\mathrm{sd}\big(\max_K Z\big)$ and $DSR = \Phi\!\left(\dfrac{SR^* - SR_{0,K}}{s_{0,K}}\right)$.

Every Sharpe ratio is per day; annualised values ($\times\sqrt{252}$) are labels only.

## §3 Data

Daily net returns minus BIL for the 23 runs of notebooks 01–07 and 60/40, rebuilt with notebook 08's constructors; their Sharpe ratios match notebook 08's in all three windows (`tests/test_reproduction.py` checks this).

In [ ]:
data = h.build_data()
X, XW, split_ts = data["X"], data["XW"], data["split_ts"]
assert [len(Xw) for Xw in XW.values()] == [4338, 3504, 834]

Sample moments of the eight core runs and 60/40 per window; Sharpe ratios are per day, with the annualized value alongside.

In [ ]:
mom = h.moments_table(XW).set_index(["run", "window"]).rename(columns={
    "SR_d": "Sharpe (daily)", "SR_ann": "Sharpe (annual)", "skew": "skewness", "kurt": "kurtosis", "rho": "autocorrelation"})
mom_shown = ml.reporting.show_table(mom, dp=2, int_cols=["T"],
                                    col_dp={"Sharpe (daily)": 4, "kurtosis": 1, "autocorrelation": 3})
mom_shown

**Reading.** Daily excess returns are fat-tailed and negatively skewed over the full window: skewness −1.25 (ERC(S)) … −0.18 (60/40), Pearson kurtosis 10.1 (BL(0.1,S)) … 22.2 (BMV(0.3)). Lag-1 autocorrelation is small, −0.085 (BMV(0.3)) … +0.012 (MaxSharpe(S)). Daily Sharpe ratios are 0.039 (GMV(S)) … 0.057 (60/40), 0.62 … 0.91 annualised.

## §4 D1: PSR and MinTRL against zero

For each run and window: the standard deviation of the daily Sharpe ratio under the paper's general formula, under iid Normal returns and by Newey–West; the PSR against cash, general and iid; and the MinTRL in years at α = 0.05.

In [ ]:
d1 = h.d1_table(XW)
d1s = d1.set_index(["run", "window"])[["T", "sd_gen", "sd_iid", "gen_over_iid", "sd_NW", "gen_over_NW", "PSR0", "PSR0_iid", "MinTRL_yrs"]].rename(columns={
    "sd_gen": "sd general", "sd_iid": "sd iid", "gen_over_iid": "general / iid", "sd_NW": "sd Newey–West",
    "gen_over_NW": "general / Newey–West", "PSR0": "PSR general", "PSR0_iid": "PSR iid", "MinTRL_yrs": "MinTRL (years)"})
d1_shown = ml.reporting.show_table(d1s, dp=2, int_cols=["T"],
                                   col_dp={"sd general": 4, "sd iid": 4, "sd Newey–West": 4, "PSR general": 4,
                                           "PSR iid": 4, "MinTRL (years)": 1})
display(d1_shown)

test_yrs = len(XW["test (> split)"]) / ml.TRADING_DAYS
long_enough = d1.loc[d1["test_long_enough"].eq(True), "run"]
print(f"test window ({test_yrs:.1f} years) at least as long as the full-window MinTRL: {', '.join(long_enough)}")

**Reading.** Every run beats cash over the full window, and fat tails barely change that. With a daily Sharpe ratio near 0.05, skewness and kurtosis barely move its variance: over the full window the general standard deviation is 0.95 (60/40) … 1.03 (MaxSharpe(S)) times the iid value and 0.89 … 1.04 times the Newey–West value. Against cash every run is significant over the full window (PSR 0.9960 for BL(0.1,S) … 1.0000 for 60/40) and the train window (PSR ≥ 0.9864). In the 3.3-year test window 6 of the 9 reach PSR ≥ 0.95 (EW only just, at 0.9501); GMV(S) 0.9208, HRP(S) 0.9451 and BL(0.1,S) 0.8967 do not.

MinTRL in years per run at full-window moments, with the lengths of the test and full windows.

In [ ]:
fig = h.fig_mintrl(d1)
plt.show()

**Reading.** The test window is too short for most runs: at their full-window Sharpe ratios, the runs need 2.9 years (60/40) to 6.6 years (BL(0.1,S), GMV(S)) of data to show a positive Sharpe ratio. Only 60/40 and BMV(0.3) (3.1 years) need less than the 3.3-year test window.

## §5 D2: power against zero, and years to 80%

Power of $H_0: SR = 0$ (one-sided, $\alpha = 0.05$) against annual $SR_1 \in \{0.25, 0.5, 0.75\}$, at each window's own $T$ and moments, and the smallest $T$ (in years) that gives 80% power. Paper formulas (eq. 15–16), on daily Sharpe ratios $SR_1/\sqrt{252}$.

In [ ]:
d2 = h.d2_table(XW)
d2["SR1"] = "SR1 = " + d2["SR1_ann"].map("{:g}".format)
pw = d2.pivot(index="run", columns=["window", "SR1"], values="power").reindex(h.RUNS).rename_axis(index="power (%)", columns=[None, None])
yr = d2.pivot(index="run", columns=["window", "SR1"], values="T80_yrs").reindex(h.RUNS).rename_axis(index="years to 80% power", columns=[None, None])
display(ml.reporting.show_table(pw, pct=list(pw.columns), int_cols=list(pw.columns)))
ml.reporting.show_table(yr, dp=1)

**Reading.** Power depends on the length of the sample far more than on the method. Over the full window the nine runs have 66–73% power against a true annual Sharpe ratio of 0.5 and 27–30% against 0.25; the test window gives 23–25% against 0.5. With full-window moments, 80% power needs 21–26 years against 0.5, 9–11 against 0.75 and 84–102 against 0.25.

Power against cash as a function of years of data, at the median full-window moments of the nine runs; the band spans the runs' own moments.

In [ ]:
fig = h.fig_power_cash(XW)
plt.show()

**Reading.** The band is narrow: at 17.2 years the runs' own moments move power by at most 7.2 percentage points. In the test window even a true Sharpe ratio of 0.75 is detected only 38–43% of the time.

## §6 D3: power of notebook 08's H1 tests (our calculation, not the paper's)

Notebook 08's H1 compares each of the 7 non-EW core methods with EW by a delta-method Newey–West test of $\Delta SR$ on the full window. Given that test's standard error $se$ (annualised), a true difference $\delta$ is detected with power $\Phi(\delta/se - z) + \Phi(-\delta/se - z)$, $z = \Phi^{-1}(1 - \alpha/2)$ unadjusted, or with $\alpha/7$ in place of $\alpha$ (Bonferroni bound for the first Holm step). Years to 80% power assume $se \propto 1/\sqrt{\text{years}}$.

In [ ]:
d3 = h.d3_table(XW)
d3_shown = d3.set_index(["method", "Delta"]).rename_axis(index=["method", "true ΔSR"])[[
    "d_sr_hac", "se", "power_unadj", "power_bonf", "Y80_unadj_yrs", "Y80_bonf_yrs"]].rename(columns={
    "d_sr_hac": "ΔSR vs EW", "power_unadj": "power (%)", "power_bonf": "power at α/7 (%)",
    "Y80_unadj_yrs": "years to 80%", "Y80_bonf_yrs": "years to 80% at α/7"})
ml.reporting.show_table(d3_shown, pct=["power (%)", "power at α/7 (%)"],
                        int_cols=["power (%)", "power at α/7 (%)", "years to 80%", "years to 80% at α/7"])

**Reading.** Notebook 08's H1 tests had little power. The standard error of ΔSR against EW is 0.10 (BL(0.1,S)) … 0.26 (MaxSharpe(S)), smallest for the runs that track EW most closely. A true gap of 0.2 would be detected with probability 12% … 51% unadjusted and 3% … 24% at the Bonferroni bound; a gap of 0.3 reaches 80% power only for BL(0.1,S) (85% unadjusted). Notebook 08's NULL HOLDS verdicts for H1 therefore mean the data cannot separate these methods from EW, not that their Sharpe ratios are equal.

Years of data needed for 80% power (unadjusted, two-sided) as a function of the true ΔSR against EW, with the length of the full window.

In [ ]:
fig = h.fig_power_ew(XW)
plt.show()

**Reading.** Telling a method from EW would take decades to centuries of data: for a gap of 0.2, 80% power needs 34 years (BL(0.1,S)) to 220 years (MaxSharpe(S)); for 0.1, 137 to 879 years. Within the plotted range only BL(0.1,S), BMV(0.3) and ERC(S) cross the 17.2-year line, at gaps of about 0.28, 0.34 and 0.43.

## §7 D4: deflated Sharpe ratio of the best of 23

The 23 runs of notebooks 01–07 are the search. The best by full-window Sharpe ratio is deflated by the False Strategy Theorem (eq. 28–30) for $K = 23$ and for an effective $K$: the number of trial clusters (k-means on $\sqrt{(1-C)/2}$, silhouette quality, seed 20261002) and the effective rank of the correlation matrix (roughly, how many independent directions the 23 return series span). The variance of the trials' Sharpe ratios is $V_{all}$; the sharp-null variance $V_0$ (mean of the generalized variance at $SR = 0$: the spread the trials' Sharpe ratios would have if every true Sharpe ratio were zero) is a sensitivity.

In [ ]:
d4 = h.d4_search(XW)
print(f"best run: {d4['best']}, Sharpe {d4['sr_star'] * h.ANN:.2f} annualized ({d4['sr_star']:.4f} per day); "
      f"{d4['nK']} clusters, effective rank {d4['k_er']:.2f}")

srk = pd.DataFrame({"Sharpe (daily)": d4["srk"], "Sharpe (annual)": d4["srk_ann"]}).sort_values("Sharpe (daily)", ascending=False)
srk["cluster"] = d4["labels"].map(h.block_names(d4["labels"]))
srk_shown = ml.reporting.show_table(srk, dp=2, col_dp={"Sharpe (daily)": 4})
display(srk_shown)

DSR_CASES = {"K=23, V_all": "K = 23", "K=nK, V_all": f"K = {d4['nK']} (clusters)",
             "K=round(K_er), V_all": f"K = {round(d4['k_er'])} (effective rank)",
             "sharp-null K=23, V0": "sharp null, K = 23", "sharp-null K=nK, V0": f"sharp null, K = {d4['nK']} (clusters)"}
assert set(d4["dsr"]["case"]) == set(DSR_CASES)
dsr = d4["dsr"].set_index("case").rename(index=DSR_CASES).rename(columns={"var": "variance", "sr0K": "expected max SR0,K", "s0K": "s0,K", "dsr": "DSR"}).rename_axis(None)
dsr_shown = ml.reporting.show_table(dsr, dp=4, int_cols=["K"])
dsr_shown["variance"] = dsr["variance"].map("{:.3e}".format)
dsr_shown

**Reading.** The best run survives the search against cash. The best of the 23 runs is BMV(0.5) at 0.86 annualised, 0.0003 ahead of BMV(0.3). The 23 Sharpe ratios are bunched (0.58 … 0.86 annualised; $V_{all}$ = 2.2e-5 per day²) because the runs are highly correlated, so the expected maximum under the null is small ($SR_{0,K}$ = 0.0092 per day for K = 23) and the DSR rounds to 1.0000 in every variant that uses $V_{all}$; with the sharp-null variance it is 0.9993 (K = 23) and 0.9999 (K = 3). The best run beats cash after the search; whether it beats EW is notebook 08's H1, which §6 shows has little power.

Correlation of the 23 runs' daily excess returns, ordered by trial cluster.

In [ ]:
fig = h.fig_search(d4)
plt.show()

**Reading.** The 23 runs fall into 3 blocks: risk-based (12: GMV, MDP, ERC, HRP, IVP), balanced / EW-like (8: EW, BL, BMV, IV) and MaxSharpe (3). The effective rank is 2.67, so both estimates put the search of notebooks 01–07 at about three independent ideas, not 23: estimator and linkage variants of one method are near-duplicates.

## §8 TAKEAWAY — the data can show these methods beat cash, not which one is best

- Against cash, every run's Sharpe ratio is significant over 17 years, and the
  best of the 23 runs survives deflation for the search.
- Against EW, the data cannot decide: detecting a Sharpe gap of 0.2 with 80%
  power would take 34 to 220 years.
- The 3.3-year test window is shorter than the minimum track record of 7 of the
  9 runs, so it cannot confirm or reject much on its own.
- The 23 runs of notebooks 01–07 amount to about three independent ideas.

This is why notebooks 08–10 report NULL HOLDS: not because the methods are
equal, but because 17 years of data cannot tell them apart. No method is shown
to beat EW, and none is shown to equal it.